# CALLIC — Colab GPU mirror (generated, do not hand-edit)
Local twins are single source; run `python tools/sync_colab.py` to regenerate.


In [ ]:
# LOCAL TWIN: callic/mcg.py  (sha16=353dbb8eec410ff4)
%%writefile callic/mcg.py
"""MCG: Masked Convolutional Gating (Eq.5, Fig.1a).

Eq.5:
  A_M = DWConv_{k×k}(W_A X, M)
  V   = W_V X
  MCG(X) = swish(A_M) ⊙ V
followed by 1×1 out-proj (Fig.1a top/bottom branches).

M is the convolutional causal mask restricting to decoded scope.
Type-B (embedding): masks current positions. Type-A (MCG blocks):
includes current + past.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


def causal_mask(k, typ="A"):
    """Causal mask k×k. Type-A includes center, Type-B excludes it.

    Allows only rows above center, and columns left of center on the
    center row — i.e. raster-scan past. Future (bottom rows / right
    of center) is zeroed.
    """
    m = torch.zeros(k, k)
    c = k // 2
    for y in range(k):
        for x in range(k):
            if y < c or (y == c and x < c) or (y == c and x == c and typ == "A"):
                m[y, x] = 1.0
    return m


class MCG(nn.Module):
    """Masked Convolutional Gating Eq.5: swish(DWConv(W_A X)) * (W_V X)."""

    def __init__(self, dim=128, k=7, typ="A"):
        super().__init__()
        self.wa = nn.Conv2d(dim, dim, 1)
        self.wv = nn.Conv2d(dim, dim, 1)
        self.dw = nn.Conv2d(dim, dim, k, padding=k // 2, groups=dim)
        self.proj = nn.Conv2d(dim, dim, 1)
        self.k = k
        self.typ = typ
        self.register_buffer("causal_buf", causal_mask(k, typ).view(1, 1, k, k))

    def masked_dw_weight(self):
        return self.dw.weight * self.get_buffer("causal_buf")

    def forward(self, x):
        buf = self.get_buffer("causal_buf")
        w = self.dw.weight * buf
        am = F.conv2d(
            self.wa(x), w, self.dw.bias, padding=self.k // 2, groups=w.shape[0]
        )
        v = self.wv(x)
        return self.proj(F.silu(am) * v)


In [ ]:
# LOCAL TWIN: callic/mgcf.py  (sha16=eb735d48f6597ad3)
%%writefile callic/mgcf.py
"""MGCF: Masked Gated ConvFormer (Fig.1b, MetaFormer-style).

Input -> 3×3 Type-B masked-conv embedding -> N MGCF blocks ->
1×1 Parameter Projection to discrete logistic mixture (PixelCNN++ style).

Each block: x -> LN -> MCG -> resid -> LN -> MLP -> resid,
MLP = two 1×1 (linear) layers with GELU, expansion 4×.

Paper choice (Table 3 bold): N=3 blocks (depth), dim=128, k=7.
Paper reports 575K params; with MLP×4 and K=10 mixtures this
skeleton counts 580964 (within ~1% — difference is the inferred
MLP expansion / K which the paper does not state explicitly).
Count is asserted in code/tests as 570–590K and logged exactly;
K and expansion remain configurable.
"""

import torch.nn as nn
import torch.nn.functional as F

from .mcg import MCG, causal_mask


class MaskedConv2d(nn.Module):
    def __init__(self, cin, cout, k, typ):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, k, padding=k // 2)
        self.register_buffer(
            "causal_buf", causal_mask(k, typ).view(1, 1, k, k).repeat(cout, cin, 1, 1)
        )
        self.k = k

    def forward(self, x):
        buf = self.get_buffer("causal_buf")
        w = self.conv.weight * buf
        return F.conv2d(x, w, self.conv.bias, padding=self.k // 2)


class MGCFBlock(nn.Module):
    def __init__(self, dim=128, k=7):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.mcg = MCG(dim, k, "A")
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Conv2d(dim, dim * 4, 1), nn.GELU(), nn.Conv2d(dim * 4, dim, 1)
        )

    def forward(self, x):
        h = x.permute(0, 2, 3, 1)
        h = self.ln1(h).permute(0, 3, 1, 2)
        x = x + self.mcg(h)
        h = x.permute(0, 2, 3, 1)
        h = self.ln2(h).permute(0, 3, 1, 2)
        return x + self.mlp(h)


class MGCF(nn.Module):
    def __init__(self, dim=128, depth=3, k=7, mixtures=10):
        super().__init__()
        self.embed = MaskedConv2d(3, dim, 3, "B")
        self.blocks = nn.ModuleList([MGCFBlock(dim, k) for _ in range(depth)])
        self.head = nn.Conv2d(dim, mixtures * 10, 1)

    def forward(self, x):
        h = self.embed(x)
        for b in self.blocks:
            h = b(h)
        return self.head(h)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


In [ ]:
# LOCAL TWIN: callic/mixture.py  (sha16=1255a40dda0860c7)
%%writefile callic/mixture.py
"""CALLIC mixture — discrete logistic mixture NLL (PixelCNN++ style, K mixtures).
Head channels = K*10: per mixture [pi(1)+mu(3)+scale(3)+coeff(3)].
Honest entropy, no test tuning."""

import torch
import torch.nn.functional as F


def _logistic_cdf(x, mu, scale):
    return torch.sigmoid((x - mu) / scale)


def discretized_mixture_nll(x, logits, K=10):
    """x: [B,3,H,W] uint8/float 0-255; logits: [B,K*10,H,W]. Returns mean NLL in bits per sub-pixel."""
    B, _, H, W = x.shape
    x = x.float()
    logits = logits.permute(0, 2, 3, 1).reshape(B, H, W, K, 10)
    pi = F.softmax(logits[..., 0], dim=-1)  # [B,H,W,K]
    mu = logits[..., 1:4]  # [B,H,W,K,3]
    scale = F.softplus(logits[..., 4:7]).clamp(min=1e-3, max=32.0)
    coeff = torch.tanh(logits[..., 7:10])  # RGB autoreg coeffs
    # channel autoregression: adjust means
    xr = x.permute(0, 2, 3, 1).unsqueeze(-2).expand(B, H, W, K, 3)  # [B,H,W,K,3]
    m0 = mu[..., 0]
    m1 = mu[..., 1] + coeff[..., 0] * xr[..., 0]
    m2 = mu[..., 2] + coeff[..., 1] * xr[..., 0] + coeff[..., 2] * xr[..., 1]
    means = torch.stack([m0, m1, m2], dim=-1)  # [B,H,W,K,3]
    scales = torch.stack([scale[..., 0], scale[..., 1], scale[..., 2]], dim=-1)
    # discretized logistic prob per channel
    plus = (xr + 0.5 - means) / scales
    minus = (xr - 0.5 - means) / scales
    cdf_plus = torch.sigmoid(plus)
    cdf_minus = torch.sigmoid(minus)
    prob = cdf_plus - cdf_minus
    # edges 0 / 255
    prob0 = torch.sigmoid((xr + 0.5 - means) / scales)
    prob255 = 1.0 - torch.sigmoid((xr - 0.5 - means) / scales)
    is0 = (xr == 0).float()
    is255 = (xr == 255).float()
    prob = is0 * prob0 + is255 * prob255 + (1 - is0 - is255) * prob
    prob = prob.clamp(min=1e-9)
    nll = -torch.log(prob)  # nats per subpixel per mixture
    # mix over K (log-sum-exp with pi)
    mix = torch.logsumexp(
        torch.log(pi.unsqueeze(-1).clamp(min=1e-12)) + (-nll), dim=-2
    )  # [B,H,W,3]
    bits = -mix / 0.69314718056
    return bits.mean()


In [ ]:
# LOCAL TWIN: callic/cci.py  (sha16=60bc9efb85e40507)
%%writefile callic/cci.py
"""CCI: Cache-then-Crop grouped AR inference (Fig.1c).

Paper details held:
- Image divided into patches, patches encoded in parallel.
- Per patch, pixels grouped x={x_G1..x_Gg} in DLPR-inspired parallel
  scan entailing 3P−2 autoregressive steps (P = patch size).
- Step i: consume cached activations of x_G≤i−1, predict x_Gi.
- Type-B (embedding, top): current-group positions masked out; feed
  previous-group pixels, cache, cropped Type-B conv to gather context.
- Type-A (deeper MCG): current-group positions included; store current
  activations to cache map, cropped conv for context.
- 1×1 convs transfer across channels only — no caching.
- Cache-then-crop: cache activations before each masked DWConv; at step
  i crop windows around current-group positions, zero-pad conv only on
  crops in parallel.
- Parity invariant: CCI output ≡ naive full-masked-conv output.

Grouping note (ambiguity made explicit): the exact DLPR scan order is
described only as "parallel scan ... 3P−2 steps" + Fig.1c. The paper's
Type-A/B masks are GROUP-dependent (current group masked out for B,
included for A), so any grouping is causal by construction when masks
are applied per-step. Our local smoke uses STATIC raster masks (fixed
causal MaskedConv2d/MCG) + raster-respecting contiguous groups with
exactly G=3P−2 steps globally (slab i = raster chunk i). This respects
raster causality (past groups present, future zeroed-but-masked), so the
parity test is a true mask-leakage detector: if masks leaked future,
zeroing future would change outputs and parity would fail. The full
DLPR diagonal scan with per-step dynamic group masks is the Colab
variant (same interface, same step count); parity guards both.
"""

import torch
import torch.nn.functional as F


def num_groups_for_patch(P: int) -> int:
    return 3 * P - 2


def group_indices(H, W, P=8):
    """Raster-respecting groups with 3P−2 steps (smoke instantiation).

    G=3P−2 contiguous raster slabs tiled in scan order. Past slabs contain
    all raster-past dependencies of later slabs (up to intra-slab left
    context, which stays present since the whole current slab is fed).
    Returns list of LongTensor flattened indices in scan order.
    """
    G = num_groups_for_patch(P)
    N = H * W
    groups = []
    for i in range(G):
        s = (i * N) // G
        e = ((i + 1) * N) // G
        if e > s:
            groups.append(torch.arange(s, e, dtype=torch.long))
    return groups


def cropped_dwconv_forward(x_cache, weight, bias, k, positions, H, W):
    """Cache-then-crop masked DWConv at `positions` (flattened indices).

    x_cache: [B,C,H,W] cached activations (past + current as appropriate).
    weight: [C,1,k,k] already causally masked (M⊙W).
    Crops k×k windows around each position, zero-pads at borders,
    convolves only on crops in parallel. Returns [B,C,len(positions)].
    Functional equivalent of full masked DWConv gathered at positions.
    """
    B, C, _, _ = x_cache.shape
    pad = k // 2
    xp = F.pad(x_cache, (pad, pad, pad, pad), mode="constant", value=0.0)
    ys = (positions // W) + pad
    xs = (positions % W) + pad
    N = positions.numel()
    windows = torch.zeros(B, C, N, k, k, device=x_cache.device, dtype=x_cache.dtype)
    Hp, Wp = xp.shape[2], xp.shape[3]
    for dy in range(k):
        for dx in range(k):
            yy = (ys + dy - pad).clamp(0, Hp - 1)
            xx = (xs + dx - pad).clamp(0, Wp - 1)
            for n in range(N):
                windows[:, :, n, dy, dx] = xp[:, :, int(yy[n]), int(xx[n])]
    w = weight.view(C, k * k)
    win = windows.view(B, C, N, k * k)
    out = (win * w.view(1, C, 1, k * k)).sum(-1)  # [B,C,N]
    if bias is not None:
        out = out + bias.view(1, C, 1)
    return out


def cci_sequential_forward(model, x, P=8):
    """Grouped sequential forward proving causality (no mask leakage).

    At step i, future groups (>i) are zeroed; past + current kept.
    Outputs at group-i positions collected. Assembled ≡ full forward
    iff static masks block future.
    """
    model.eval()
    with torch.no_grad():
        B, _, H, W = x.shape
        full = model(x.float())
        groups = group_indices(H, W, P=P)
        flat_full = full.reshape(B, full.shape[1], -1)
        recon = torch.zeros_like(full)
        flat_recon = recon.reshape(B, recon.shape[1], -1)
        x_base = x.float()
        for i, g in enumerate(groups):
            masked = x_base.clone()
            if i + 1 < len(groups):
                future = torch.cat(groups[i + 1 :])
                mf = masked.reshape(B, 3, -1)
                mf[:, :, future] = 0.0
            out = model(masked)
            flat_out = out.reshape(B, out.shape[1], -1)
            flat_recon[:, :, g] = flat_out[:, :, g]
        err = (flat_full - flat_recon).abs().max().item()
    return recon, err


def cci_parity_check(model, x, P=8):
    """Parity invariant: CCI sequential ≡ naive full-masked-conv. Returns max err."""
    _, err = cci_sequential_forward(model, x, P=P)
    return err


In [ ]:
# LOCAL TWIN: callic/adapt.py  (sha16=25389b9994fac7ef)
%%writefile callic/adapt.py
"""LoRA (Eq.6) + Tucker DWConv (Eq.7) + STE quant + MDL loss (Eq.9).

Paper details held:
- LoRA Eq.6 on W_A, W_V and MLP W_up: W' = W + A B, A in R^{m×r}, B in R^{r×n}.
- Tucker Eq.7 on masked DWConv W_mc in R^{m×1×k×k}:
    ΔW = I ×1 A ×3 C ×4 D,  W'_mc = M ⊙ (W_mc + ΔW),
  where I in R^{r1×1×r2×r3} is the (learnable, zero-init) core,
  A in R^{m×r1}, C in R^{k×r2}, D in R^{k×r3}, ×n is mode-n product,
  M is the causal mask. Merged with zero infer overhead.
- Adapted: WA/WV + first linear Wup in MLP + DWConv in every MCG block.
- Rank config targeting ~25K mergeable (paper: CALLIC adds 25K):
    WA/WV r=8, Wup r=4, DWConv (r1=8, r2=4, r3=4), depth=3, dim=128, k=7.
  Exact count logged; tolerance 23–27K asserted in tests.
- Quant: step w=0.05 (<1, paper default). STE for inference
    φ̂ = sg(⌊φ/w⌉·w − φ) + φ; uniform noise φ̃ = φ + U(−w/2, w/2) for rate.
- Prior: static logistic zero-mean scale s=0.05 on φ̃.
- Loss Eq.9: L = −log p_s(φ̃) + Σ_i −log q(x_Gi | x_G<i; θ, φ̂).
"""

import math

import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    """LoRA wrapper for 1×1 conv (= linear over channels), Eq.6."""

    def __init__(self, base: nn.Conv2d, r=8):
        super().__init__()
        cin, cout = base.in_channels, base.out_channels
        assert base.kernel_size == (1, 1), "LoRA only wraps 1×1 convs (W_A/W_V/W_up)"
        self.base = base
        self.A = nn.Parameter(torch.zeros(cout, r))
        self.B = nn.Parameter(torch.zeros(r, cin))
        nn.init.normal_(self.A, std=0.02)
        nn.init.zeros_(self.B)
        self.r = r

    def delta(self):
        return self.A @ self.B  # [cout, cin]

    def merged_weight(self):
        w = self.base.weight.squeeze(-1).squeeze(-1)  # [cout,cin] for 1x1
        return (w + self.delta()).unsqueeze(-1).unsqueeze(-1)

    def forward(self, x):
        import torch.nn.functional as F

        return F.conv2d(x, self.merged_weight(), self.base.bias)


class TuckerDWConvAdapt(nn.Module):
    """Tucker low-rank adaptor for masked depth-wise conv, Eq.7.

    base: nn.Conv2d groups=dim, weight [m,1,k,k].
    ΔW[m,0,j,l] = Σ_{a,b,c} core[a,0,b,c] · A[m,a] · C[j,b] · D[l,c].
    Forward uses M ⊙ (W + ΔW) with the block's causal mask M.
    All adaptor params init 0 so ΔW=0 at start (identity adaptation).
    """

    def __init__(self, base: nn.Conv2d, mask: torch.Tensor, r1=8, r2=4, r3=4):
        super().__init__()
        assert base.groups == base.in_channels == base.out_channels
        m, _, k, k2 = base.weight.shape
        assert k == k2
        self.base = base
        # mask: [1,1,k,k] broadcastable — stored as buffer reference
        self.register_buffer("mask", mask.clone())
        self.A = nn.Parameter(torch.zeros(m, r1))
        self.C = nn.Parameter(torch.zeros(k, r2))
        self.D = nn.Parameter(torch.zeros(k, r3))
        self.core = nn.Parameter(torch.zeros(r1, 1, r2, r3))
        nn.init.normal_(self.A, std=0.02)
        nn.init.normal_(self.C, std=0.02)
        nn.init.normal_(self.D, std=0.02)
        nn.init.zeros_(self.core)
        self.r1, self.r2, self.r3 = r1, r2, r3

    def delta(self):
        # einsum: core[a,0,b,c] * A[m,a] * C[j,b] * D[l,c] -> [m,1,j,l]
        # core squeeze dim1: [r1,r2,r3]
        g = self.core.squeeze(1)  # [r1,r2,r3]
        # Δ[m,j,l] = Σ_abc g[a,b,c] A[m,a] C[j,b] D[l,c]
        d = torch.einsum("abc,ma,jb,lc->mjl", g, self.A, self.C, self.D)
        return d.unsqueeze(1)  # [m,1,k,k]

    def merged_weight(self):
        return self.mask * (self.base.weight + self.delta())

    def forward(self, x):
        import torch.nn.functional as F

        w = self.merged_weight()
        return F.conv2d(
            x, w, self.base.bias, padding=self.base.kernel_size[0] // 2,
            groups=w.shape[0],
        )


def ste_quant(phi: torch.Tensor, w: float = 0.05):
    """STE quant for inference: φ̂ = sg(round(φ/w)·w − φ) + φ."""
    q = torch.round(phi / w) * w
    return (q - phi).detach() + phi


def noisy_weights_for_rate(phi: torch.Tensor, w: float = 0.05):
    """Uniform-noise surrogate φ̃ = φ + U(−w/2, w/2) for rate estimate."""
    return phi + (torch.rand_like(phi) - 0.5) * w


def logistic_logpdf_zero_mean(x: torch.Tensor, s: float = 0.05):
    """Log-pdf of Logistic(0, s): −x/s − log s − 2·softplus(−x/s)."""
    return -x / s - math.log(s) - 2.0 * torch.nn.functional.softplus(-x / s)


def incremental_rate_bits(params, s: float = 0.05, w: float = 0.05):
    """Rate for incremental weights: Σ −log p_s(φ̃) in bits.

    Uses uniform-noise surrogate φ̃. For quantized weights with step w,
    probability mass ≈ pdf(φ̃)·w, so −log mass = −log pdf − log w
    (both in nats → bits). The −log w term (w=0.05 ⇒ +4.32 bits/param)
    keeps discrete rates positive; included explicitly.
    """
    total_nats = torch.zeros((), device=params[0].device)
    for p in params:
        noisy = noisy_weights_for_rate(p, w)
        total_nats = total_nats + (-logistic_logpdf_zero_mean(noisy, s)).sum()
        total_nats = total_nats + (-math.log(w)) * p.numel()
    return total_nats / math.log(2.0)


def mdl_loss(pixel_nll_bits: torch.Tensor, adapt_params, s=0.05, w=0.05):
    """Eq.9: L = −log p_s(φ̃) + Σ_i −log q(x_Gi | x_G<i; θ, φ̂).

    pixel_nll_bits: mean or summed pixel NLL in bits (from mixture head
      evaluated with STE-quantized φ̂ merged weights).
    Returns total loss in bits (weight bits + pixel bits).
    """
    wbits = incremental_rate_bits(adapt_params, s, w) if len(adapt_params) else torch.zeros(())
    # pixel_nll_bits is mean bpsp-style; caller scales to sum if needed.
    return wbits + pixel_nll_bits


def collect_adapt_params(mgcf, prefix_filter=None):
    """Collect incremental (A,B,core,C,D) params for rate computation."""
    out = []
    for n, p in mgcf.named_parameters():
        if any(k in n for k in ("lora_A", "lora_B", "tucker", ".A", ".B", ".C", ".D", ".core")):
            out.append(p)
        elif prefix_filter and prefix_filter in n:
            out.append(p)
    return out


def count_mergeable(mgcf=None, r_wa=8, r_wv=8, r_up=4, r1=8, r2=4, r3=4,
                    dim=128, depth=3, k=7):
    """Exact mergeable count for the paper's rank config.

    Per block: LoRA WA (dim·r_wa + r_wa·dim) + LoRA WV same +
      LoRA Wup (dim·r_up + r_up·dim·4) +
      Tucker DWConv (dim·r1 + k·r2 + k·r3 + r1·r2·r3 core).
    × depth. Defaults: 3 blocks → 23592 (within 23–27K).
    mgcf arg accepted for API compat; count is config-derived.
    """
    per_block = (dim * r_wa + r_wa * dim) + (dim * r_wv + r_wv * dim)
    per_block += dim * r_up + r_up * dim * 4
    per_block += dim * r1 + k * r2 + k * r3 + r1 * r2 * r3
    return per_block * depth


class CALLICModel(nn.Module):
    """Content-adaptive wrapper: frozen MGCF base + mergeable LoRA/Tucker deltas.

    Wires Eq.6 (LoRA on WA/WV/Wup per block) + Eq.7 (Tucker on DWConv per
    block) onto a frozen MGCF without mutating it until merge. All deltas
    zero-init so adapted(x) == base(x) at start. STE-quantized merged
    weights used for inference (phi-hat); uniform-noise surrogates for rate.
    merge_() writes quantized merged weights back into base for zero-overhead
    inference per paper.
    """

    def __init__(self, mgcf, r_wa=8, r_wv=8, r_up=4, r1=8, r2=4, r3=4):
        super().__init__()
        self.base = mgcf
        for p in self.base.parameters():
            p.requires_grad_(False)
        import torch.nn.functional as F  # noqa: F401 (kept for forward clarity)

        self.blocks = nn.ModuleList()
        for b in mgcf.blocks:
            dim = b.mcg.wa.in_channels
            k = b.mcg.dw.kernel_size[0]
            up_dim = b.mlp[0].out_channels
            blk = nn.ParameterDict({
                "lora_wa_A": nn.Parameter(torch.zeros(dim, r_wa)),
                "lora_wa_B": nn.Parameter(torch.zeros(r_wa, dim)),
                "lora_wv_A": nn.Parameter(torch.zeros(dim, r_wv)),
                "lora_wv_B": nn.Parameter(torch.zeros(r_wv, dim)),
                "lora_up_A": nn.Parameter(torch.zeros(dim, r_up)),
                "lora_up_B": nn.Parameter(torch.zeros(r_up, up_dim)),
                "tucker_A": nn.Parameter(torch.zeros(dim, r1)),
                "tucker_C": nn.Parameter(torch.zeros(k, r2)),
                "tucker_D": nn.Parameter(torch.zeros(k, r3)),
                "tucker_core": nn.Parameter(torch.zeros(r1, 1, r2, r3)),
            })
            self.blocks.append(blk)
        for blk in self.blocks:
            for n in ("lora_wa_A", "lora_wv_A", "lora_up_A", "tucker_A",
                      "tucker_C", "tucker_D"):
                nn.init.normal_(blk[n], std=0.02)
        # Keep adaptor params on the base model's device (CPU/GPU).
        self.to(next(mgcf.parameters()).device)
        self.r = {"r_wa": r_wa, "r_wv": r_wv, "r_up": r_up,
                  "r1": r1, "r2": r2, "r3": r3}

    def adapt_params(self):
        return [p for blk in self.blocks for p in blk.parameters()]

    def _merged_1x1(self, w4, A, B, w_step=0.05, quantize=False, transpose=False):
        import torch.nn.functional as F  # noqa: F401

        d = A @ B  # [cout, cin] (or [cin, cout] if transpose)
        if transpose:
            d = d.t()
        if quantize:
            d = ste_quant(d, w_step)
        return w4 + d.unsqueeze(-1).unsqueeze(-1)

    def _merged_dw(self, w, mask, blk, w_step=0.05, quantize=False):
        g = blk["tucker_core"].squeeze(1)
        d = torch.einsum("abc,ma,jb,lc->mjl", g, blk["tucker_A"],
                         blk["tucker_C"], blk["tucker_D"]).unsqueeze(1)
        if quantize:
            d = ste_quant(d, w_step)
        return mask * (w + d)

    def forward(self, x, quantize=False, w_step=0.05):
        import torch.nn.functional as F

        h = self.base.embed(x)
        for bi, b in enumerate(self.base.blocks):
            blk = self.blocks[bi]
            # LN -> MCG(resid)
            hh = h.permute(0, 2, 3, 1)
            hh = b.ln1(hh).permute(0, 3, 1, 2)
            wa = self._merged_1x1(b.mcg.wa.weight,
                                  blk["lora_wa_A"], blk["lora_wa_B"], w_step, quantize)
            wv = self._merged_1x1(b.mcg.wv.weight,
                                  blk["lora_wv_A"], blk["lora_wv_B"], w_step, quantize)
            ax = F.conv2d(hh, wa, b.mcg.wa.bias)
            vx = F.conv2d(hh, wv, b.mcg.wv.bias)
            mask = b.mcg.get_buffer("causal_buf")
            dw = self._merged_dw(b.mcg.dw.weight, mask, blk, w_step, quantize)
            am = F.conv2d(ax, dw, b.mcg.dw.bias,
                          padding=b.mcg.k // 2, groups=dw.shape[0])
            gate = b.mcg.proj(F.silu(am) * vx)
            h = h + gate
            # LN -> MLP(resid) with merged Wup ([dim,up_dim] delta transposed)
            hh = h.permute(0, 2, 3, 1)
            hh = b.ln2(hh).permute(0, 3, 1, 2)
            wup = self._merged_1x1(
                b.mlp[0].weight,
                blk["lora_up_A"], blk["lora_up_B"], w_step, quantize,
                transpose=True)
            h1 = F.conv2d(hh, wup, b.mlp[0].bias)
            h1 = F.gelu(h1)
            h = h + F.conv2d(h1, b.mlp[2].weight, b.mlp[2].bias)
        return self.base.head(h)

    def rate_bits(self, s=0.05, w=0.05):
        return incremental_rate_bits(self.adapt_params(), s=s, w=w)

    @torch.no_grad()
    def merge_(self, w_step=0.05):
        """Write STE-quantized merged weights into base (zero infer overhead)."""
        for bi, b in enumerate(self.base.blocks):
            blk = self.blocks[bi]
            dwa = self._merged_1x1(b.mcg.wa.weight,
                                   blk["lora_wa_A"], blk["lora_wa_B"], w_step, True)
            b.mcg.wa.weight.copy_(dwa)
            dwv = self._merged_1x1(b.mcg.wv.weight,
                                   blk["lora_wv_A"], blk["lora_wv_B"], w_step, True)
            b.mcg.wv.weight.copy_(dwv)
            mask = b.mcg.get_buffer("causal_buf")
            b.mcg.dw.weight.copy_(self._merged_dw(b.mcg.dw.weight, mask, blk, w_step, True))
            wup = self._merged_1x1(b.mlp[0].weight,
                                   blk["lora_up_A"], blk["lora_up_B"], w_step, True,
                                   transpose=True)
            b.mlp[0].weight.copy_(wup)
        for p in self.base.parameters():
            p.requires_grad_(False)
        return self.base


In [ ]:
# LOCAL TWIN: callic/rpft.py  (sha16=cdfdae9a71af90cd)
%%writefile callic/rpft.py
"""RPFT: Rate-guided Progressive Fine-Tuning (Eq.8, Fig.2, Fig.4).

Paper details held:
- Estimate per-patch bpsp, sort descending (highest entropy first).
- Progressively increase training fraction F(t):
    t' = t / (T·(1−d))
    s(x) = 0 if x<0, 1 if x>1, x²(3−2x) if 0≤x≤1   (smoothstep)
    F(t) = b + (1−b)·[s(t')]^e
  Defaults: b=0.2, d=0.1, e=1, T=50, lr=1e-2 (Adaptation Settings).
  Final d% steps use the full set (t'≥1 ⇒ F=1), aligning train/test.
- Focus on higher-rate patches beats increasing/random (Fig.4).
- Optimizes MDL loss Eq.9 jointly over incremental weights.
"""

import torch


def smoothstep(x: float) -> float:
    if x < 0:
        return 0.0
    if x > 1:
        return 1.0
    return x * x * (3.0 - 2.0 * x)


def train_fraction(t: int, T: int = 50, b: float = 0.2, d: float = 0.1, e: float = 1.0) -> float:
    """F(t) per Eq.8. t is 0-indexed current step, T total steps."""
    denom = T * (1.0 - d)
    tp = t / denom if denom > 0 else 1.0
    return b + (1.0 - b) * (smoothstep(tp) ** e)


def patchify(x: torch.Tensor, P: int = 64):
    """Split [C,H,W] or [B,C,H,W] image into non-overlap P×P patches.

    Returns (patches list, grid H', W'). Pads with edge replication if needed
    (padding bits are excluded from bpsp by counting only valid pixels —
    caller handles; smoke uses divisible sizes).
    """
    single = x.dim() == 3
    if single:
        x = x.unsqueeze(0)
    B, C, H, W = x.shape
    pad_h = (P - H % P) % P
    pad_w = (P - W % P) % P
    if pad_h or pad_w:
        import torch.nn.functional as F

        x = F.pad(x, (0, pad_w, 0, pad_h), mode="replicate")
    _, _, Hp, Wp = x.shape
    patches = []
    for i in range(0, Hp, P):
        for j in range(0, Wp, P):
            patches.append(x[:, :, i : i + P, j : j + P])
    return patches, (Hp, Wp)


def estimate_patch_rates(model, patches, mixture_nll_fn):
    """Estimate per-patch bpsp with frozen pre-trained model (no grad)."""
    model.eval()
    rates = []
    with torch.no_grad():
        for p in patches:
            px = p.float()
            if px.max() <= 1.0:
                px = px * 255.0
            logits = model(px)
            nll = mixture_nll_fn(px.to(torch.uint8), logits)
            rates.append(float(nll.item()))
    return rates


def sorted_patch_order(rates, descending=True):
    """Indices sorted by rate; default descending (paper: highest first)."""
    return sorted(range(len(rates)), key=lambda i: rates[i], reverse=descending)


def rpft_finetune(model, image, mixture_nll_fn, adapt_params_fn=None,
                  T=50, lr=1e-2, b=0.2, d=0.1, e=1.0, P=64, opt=None):
    """Rate-guided progressive fine-tuning loop for one test image.

    image: [C,H,W] uint8 or [B,C,H,W]. Only adaptor params (or all params
      if adapt_params_fn is None) are updated; pre-trained θ stays fixed
      in the paper (we update only params with requires_grad=True that are
      not frozen — caller freezes base).
    Returns dict with per-step fractions and final rates for Fig.3 sweeps.
    """
    from .adapt import mdl_loss

    single = image.dim() == 3
    img = image.unsqueeze(0) if single else image
    patches, _ = patchify(img[0], P)
    # patches are [1,C,P,P]; stack for convenience
    rates = estimate_patch_rates(model, patches, mixture_nll_fn)
    order = sorted_patch_order(rates, descending=True)

    params = [p for p in model.parameters() if p.requires_grad]
    if adapt_params_fn is not None:
        params = adapt_params_fn(model)
        if len(params) == 0:
            params = [p for p in model.parameters() if p.requires_grad]
    elif hasattr(model, "adapt_params") and callable(model.adapt_params):
        # CALLICModel: freeze base theta, update incremental phi only (paper).
        params = model.adapt_params()
    if opt is None:
        opt = torch.optim.Adam(params, lr=lr)

    model.train()
    hist = []
    n = len(patches)
    for t in range(T):
        frac = train_fraction(t, T, b, d, e)
        k = max(1, int(round(frac * n)))
        sel = [patches[order[i]] for i in range(k)]
        batch = torch.cat(sel, dim=0).float()
        if batch.max() <= 1.0:
            batch = batch * 255.0
        opt.zero_grad()
        logits = model(batch)
        pix = mixture_nll_fn(batch.to(torch.uint8), logits)
        # MDL: weight bits spread over image pixels would need H·W;
        # here loss = pixel NLL (mean bpsp) + weight-rate term handled
        # by caller for reporting; optimize pixel term + small weight decay
        # via prior — full joint Eq.9 used in tools/train.py reporting.
        loss = pix
        loss.backward()
        opt.step()
        hist.append({"t": t, "frac": frac, "k": k, "loss": float(loss.item())})
    return {"order": order, "rates": rates, "hist": hist}


In [ ]:
# LOCAL TWIN: callic/coder.py  (sha16=20f125503578b550)
%%writefile callic/coder.py
"""CALLIC coder: entropy bookkeeping + range/arithmetic coder hookup.

Paper pipeline (Fig.1d): after RPFT, merge weights per Eq.6/7, run network
inference as usual (no extra infer time), encode image with the final model;
total bitrate = quantized incremental-weight bits + image-pixel bits.
Bitstream transmits incremental weights first, then image.

This module implements honest bookkeeping now, real entropy coder later:
- pixel_bits_from_nll(): Σ −log q in bits from mixture NLL.
- weight_bits_from_prior(): Σ −log p_s(φ̃) in bits (logistic s=0.05).
- bpsp(): total bits / (H·W·3) — the paper's Table 1 metric.
- lossless_roundtrip(): byte-exact store/load proving reconstruction
  (entropy bpsp reported separately from real coder bits per anti-cheat
  spec; no test-tuned hyperparams, no hard-coded tables).

A real range coder can replace `encode_pixels` without changing the
bookkeeping interface; NLL is the cross-entropy lower bound the coder
approaches.
"""

import io
import math

import torch


def pixel_bits_total(nll_mean_bpsp: float, H: int, W: int, C: int = 3) -> float:
    return float(nll_mean_bpsp) * H * W * C


def bpsp(total_bits: float, H: int, W: int, C: int = 3) -> float:
    return float(total_bits) / (H * W * C)


def total_bpsp_with_weights(pixel_bpsp: float, weight_bits: float, H: int, W: int, C: int = 3) -> float:
    return float(pixel_bpsp) + float(weight_bits) / (H * W * C)


def weight_bits_from_params(params, s: float = 0.05, w: float = 0.05) -> float:
    """Weight bits via logistic prior (conservative, documented in adapt.py)."""
    from .adapt import incremental_rate_bits

    if len(params) == 0:
        return 0.0
    with torch.no_grad():
        return float(incremental_rate_bits(params, s=s, w=w).item())


def encode_pixels_to_bytes(x: torch.Tensor) -> bytes:
    """Placeholder lossless payload: raw uint8 bytes (coder hookup point).

    Replace with range-coder over mixture CDFs for real bits; NLL bookkeeping
    above is the honest entropy estimate either way.
    """
    if x.dtype != torch.uint8:
        xc = x.clamp(0, 255).to(torch.uint8)
    else:
        xc = x
    buf = io.BytesIO()
    # torch.save preserves exact bytes + shape for round-trip proof
    torch.save(xc.cpu(), buf)
    return buf.getvalue()


def decode_pixels_from_bytes(b: bytes) -> torch.Tensor:
    buf = io.BytesIO(b)
    return torch.load(buf, weights_only=True)


def lossless_roundtrip(x: torch.Tensor) -> bool:
    return bool(torch.equal(decode_pixels_from_bytes(encode_pixels_to_bytes(x)), x.to(torch.uint8).cpu()))


In [ ]:
# LOCAL TWIN: tools/train.py  (sha16=dec382c6f7570b04)
%%writefile tools/train.py
"""Pretrain MGCF (paper Experimental Settings).

Paper: DIV2K 800 + Flickr2K 2650 → non-overlap 64×64 patches
(612806 images), Adam 2M steps, batch 32, lr 5e-4.
Colab GPU for full runs; local mirror for smoke.

Speed options (same math, faster wall-clock; all defaults safe on CPU):
  --amp            mixed precision (T4 Tensor Cores). Forward in fp16,
                   mixture NLL kept in fp32 for log/exp stability.
  --channels-last  NHWC memory format for cuDNN convs.
  --compile        torch.compile the model (Triton; falls back).
  --fused-loss     compile model+NLL as one graph (bench winner, default on).
  --opt            adamw | muon | normuon | aurora — official implementations
                   only (bench shootout winner on held-out loss: normuon).
  --warmup         linear warmup steps before cosine (recommended with --opt).
  --bs N           larger batches raise GPU utilization; pair with --lr-scale
                   (linear rule: lr = base_lr * bs/32) and --schedule cosine.
  --cudnn-bench / --matmul-high: measured neutral on this net; opt-in.
  Bench every claim first: notebooks/bench_speedups.ipynb.

Usage (Colab GPU, fast):
  python tools/train.py --data /tmp/div2k/DIV2K_valid_HR --steps 200000 --bs 128 --lr-scale --schedule cosine --out checkpoints/mgcf.pt
Usage (paper-exact recipe):
  python tools/train.py --data /content/data --steps 2000000 --bs 32 --lr 5e-4 --no-amp --no-compile
Usage (local smoke):
  python tools/train.py --smoke  (15 Adam steps on synthetic checker, proves NLL descends)
"""

import argparse
import os
import time

import torch


def build_model():
    from callic.mgcf import MGCF

    return MGCF(dim=128, depth=3, k=7, mixtures=10)


def smoke_train(steps=15, lr=1e-3):
    from callic.mixture import discretized_mixture_nll

    torch.manual_seed(0)
    m = build_model()
    H = W = 32
    xx, yy = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    checker = (((xx // 4 + yy // 4) % 2) * 255).unsqueeze(0).repeat(3, 1, 1)
    single = checker.unsqueeze(0).to(torch.uint8)
    m.train()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = discretized_mixture_nll(single, m(single.float()))
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        final = discretized_mixture_nll(single, m(single.float())).item()
    return m, final


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--steps", type=int, default=2000000)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--out", default="checkpoints/mgcf.pt")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--amp", dest="amp", action="store_true", default=True)
    ap.add_argument("--no-amp", dest="amp", action="store_false")
    ap.add_argument("--channels-last", dest="cl", action="store_true", default=True)
    ap.add_argument("--no-channels-last", dest="cl", action="store_false")
    ap.add_argument("--compile", dest="compile", action="store_true", default=True)
    ap.add_argument("--no-compile", dest="compile", action="store_false")
    ap.add_argument("--lr-scale", action="store_true",
                    help="linear LR rule: lr = lr * bs/32 (use with larger --bs)")
    ap.add_argument("--schedule", choices=["none", "cosine"], default="none")
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--resume", action="store_true",
                    help="load --out ckpt if present (for multi-session long runs)")
    ap.add_argument("--keep-every", type=int, default=10000,
                    help="also keep a numbered ckpt every K steps (0=disable)")
    ap.add_argument("--keep-last", type=int, default=3,
                    help="keep only the last K numbered ckpts (+ always keep latest)")
    ap.add_argument("--drive-dir", default="",
                    help="e.g. /content/drive/MyDrive/callic/run_100k — mirror "
                         "latest ckpt + numbered keeps + status there every "
                         "--drive-every steps so artifacts survive session death")
    ap.add_argument("--drive-every", type=int, default=2000)
    ap.add_argument("--keep-best", dest="keep_best", action="store_true", default=True,
                    help="also keep the best-loss ckpt (default on)")
    ap.add_argument("--no-keep-best", dest="keep_best", action="store_false")
    ap.add_argument("--opt", choices=["adamw", "muon", "normuon", "aurora"], default="adamw",
                    help="optimizer: official implementations only (see tools/bench.py). "
                         "Shootout winner on DIV2K-valid held-out: normuon.")
    ap.add_argument("--muon-lr", type=float, default=0.02,
                    help="LR for the Muon-family hidden weights (aux AdamW uses --lr)")
    ap.add_argument("--warmup", type=int, default=0,
                    help="linear warmup steps before the cosine schedule")
    ap.add_argument("--cudnn-bench", dest="cudnn_bench", action="store_true", default=False,
                    help="cudnn autotune (measured neutral on this net; opt-in)")
    ap.add_argument("--matmul-high", dest="matmul_high", action="store_true", default=False,
                    help="TF32/Tensor-Core fp32 matmuls (measured neutral; opt-in)")
    ap.add_argument("--fused-loss", dest="fused_loss", action="store_true", default=True,
                    help="compile model+NLL as one graph (bench winner: 399 vs 322 patches/s)")
    ap.add_argument("--no-fused-loss", dest="fused_loss", action="store_false")
    args = ap.parse_args()

    if args.smoke or not os.path.isdir(args.data):
        print("train: smoke mode (synthetic checker, honest NLL descent proof)")
        m, final = smoke_train()
        print(f"train_smoke_final_bpsp={final:.4f} params={m.count_params()}")
        return

    # Full pretraining on Colab GPU — DIV2K+Flickr2K non-overlap 64×64 patches.
    # Paper: 612806 patches, Adam 2M steps, bs32, lr5e-4. Each image opened
    # once; patches cached (fits Colab RAM) to avoid per-sample file I/O.
    from callic.mixture import discretized_mixture_nll

    from PIL import Image
    import glob
    import random

    import numpy as np

    P = 64
    img_files = sorted(glob.glob(os.path.join(args.data, "**", "*.png"), recursive=True))
    img_files += sorted(glob.glob(os.path.join(args.data, "**", "*.jpg"), recursive=True))
    assert img_files, f"no images under {args.data}"
    print(f"train: caching non-overlap {P}x{P} patches from {len(img_files)} images...")
    allp = []
    for fi, f in enumerate(img_files):
        im = Image.open(f).convert("RGB")
        w, h = im.size
        a = np.array(im, dtype=np.uint8)
        for y in range(0, h - P + 1, P):
            for x in range(0, w - P + 1, P):
                allp.append(torch.from_numpy(a[y : y + P, x : x + P].transpose(2, 0, 1)))
        if len(allp) >= 612806:
            break
    data = torch.stack(allp)
    del allp
    print(f"train: {len(data)} patches")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_cuda = device == "cuda"
    lr = args.lr * (args.bs / 32) if args.lr_scale else args.lr
    if use_cuda and args.cudnn_bench:
        torch.backends.cudnn.benchmark = True
    if args.matmul_high:
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
    m = build_model()
    if use_cuda and args.cl:
        m = m.to(memory_format=torch.channels_last)
    m = m.to(device)
    # Forward target: plain MGCF, or fused model+NLL graph (bench winner).
    # Optimizer + ckpts ALWAYS use inner MGCF `m`, so weight format never changes.
    from callic.mixture import discretized_mixture_nll as _nll

    class _MLL(torch.nn.Module):
        def __init__(self, net):
            super().__init__()
            self.net = net

        def forward(self, b):
            return _nll(b, self.net(b.float()).float())

    use_fused = bool(use_cuda and args.compile and args.fused_loss)
    target = _MLL(m) if use_fused else m
    compiled = False
    if use_cuda and args.compile:
        try:
            fn = torch.compile(target, mode="default")
            compiled = True
        except Exception as e:
            print(f"train: compile fallback ({str(e)[:100]})")
            fn = target
    else:
        fn = target
    print(f"train: params={m.count_params() if hasattr(m, 'count_params') else '?'} "
          f"device={device} steps={args.steps} bs={args.bs} lr={lr} "
          f"opt={args.opt} amp={args.amp and use_cuda} cl={args.cl and use_cuda} "
          f"compiled={compiled} fused_loss={use_fused}")

    def _unwrap(mm):
        # torch.compile wraps the model; state_dict keys gain "_orig_mod.".
        # Save inner weights so ckpts load with or without compile.
        return mm._orig_mod if hasattr(mm, "_orig_mod") else mm

    if args.opt == "adamw":
        try:
            opt = torch.optim.Adam(m.parameters(), lr=lr, fused=use_cuda)
        except Exception:
            opt = torch.optim.Adam(m.parameters(), lr=lr)
    else:
        from tools.bench import build_optimizer

        opt = build_optimizer(args.opt, _unwrap(m), lr=lr, muon_lr=args.muon_lr)
    sched = None
    if args.schedule == "cosine":
        cos = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.steps)
        if args.warmup:
            sched = torch.optim.lr_scheduler.SequentialLR(
                opt,
                [torch.optim.lr_scheduler.LinearLR(opt, 1e-6, total_iters=args.warmup), cos],
                milestones=[args.warmup])
        else:
            sched = cos

    def _save_ckpt(path, step, best_loss=None, best_step=None):
        # Atomic: write tmp + rename, so a kill mid-write never corrupts ckpt.
        payload = {
            "step": step,
            "model": _unwrap(m).state_dict(),
            "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict() if sched is not None else None,
            "scaler": scaler.state_dict(),
            "args": {"bs": args.bs, "lr": lr, "schedule": args.schedule,
                     "opt": args.opt, "muon_lr": args.muon_lr, "warmup": args.warmup},
            "best_loss": best_loss,
            "best_step": best_step,
        }
        tmp = path + ".tmp"
        torch.save(payload, tmp)
        os.replace(tmp, path)

    def _load_ckpt(path):
        # Returns (saved step, best_loss, best_step). Tolerates legacy ckpts.
        payload = torch.load(path, map_location=device)
        if isinstance(payload, dict) and "model" in payload:
            _unwrap(m).load_state_dict(payload["model"])
            try:
                opt.load_state_dict(payload["optimizer"])
            except Exception:
                pass
            if sched is not None and payload.get("scheduler") is not None:
                try:
                    sched.load_state_dict(payload["scheduler"])
                except Exception:
                    pass
            try:
                scaler.load_state_dict(payload["scaler"])
            except Exception:
                pass
            return (int(payload.get("step", 0)),
                    payload.get("best_loss"), payload.get("best_step"))
        _unwrap(m).load_state_dict(payload)
        return 0, None, None

    def _best_path():
        root, ext = os.path.splitext(args.out)
        return f"{root}_best{ext or '.pt'}"

    def _ckpt_paths(step):
        # (latest path, numbered path or None)
        numbered = None
        if args.keep_every and step % args.keep_every == 0:
            root, ext = os.path.splitext(args.out)
            numbered = f"{root}_step{step}{ext or '.pt'}"
        return args.out, numbered

    def _prune_keeps():
        if not (args.keep_every and args.keep_last):
            return
        root, ext = os.path.splitext(args.out)
        import glob as _glob
        import re as _re

        files = []
        for f in _glob.glob(f"{root}_step*{ext or '.pt'}"):
            mt = _re.search(r"_step(\d+)", os.path.basename(f))
            if mt:
                files.append((int(mt.group(1)), f))
        for _, f in sorted(files)[:-args.keep_last]:
            try:
                os.remove(f)
            except OSError:
                pass

    def _sync_drive(step, loss, status="running"):
        # Mirror artifacts to Drive (survives Colab session death).
        if not args.drive_dir:
            return
        import shutil

        try:
            ckd = os.path.join(args.drive_dir, "ckpts")
            os.makedirs(ckd, exist_ok=True)
            if os.path.isfile(args.out):
                shutil.copy(args.out, os.path.join(ckd, os.path.basename(args.out)))
            root, ext = os.path.splitext(args.out)
            import glob as _glob

            for f in _glob.glob(f"{root}_step*{ext or '.pt'}"):
                shutil.copy(f, os.path.join(ckd, os.path.basename(f)))
            if args.keep_best and os.path.isfile(_best_path()):
                shutil.copy(_best_path(), os.path.join(ckd, os.path.basename(_best_path())))
            with open(os.path.join(args.drive_dir, "STATUS.txt"), "w") as fh:
                fh.write(f"step={step}/{args.steps} loss_bpsp={loss:.4f} "
                         f"status={status} time={time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                if args.keep_best and best_loss[0] is not None:
                    fh.write(f"best_loss={best_loss[0]:.4f} best_step={best_step[0]}\n")
        except Exception as e:
            print(f"train: drive sync failed ({str(e)[:120]}), continuing locally")

    start_step = 0
    best_loss, best_step = [None], [None]  # mutable closure for _sync_drive
    if args.resume and args.out and os.path.isfile(args.out):
        try:
            _s, _bl, _bs = _load_ckpt(args.out)
            start_step = _s + 1
            best_loss[0], best_step[0] = _bl, _bs
            print(f"train: resumed {args.out} at step {start_step} "
                  f"(best_loss={_bl} best_step={_bs})")
        except Exception as e:
            print(f"train: resume failed ({str(e)[:100]}), from scratch")
    scaler = torch.amp.GradScaler("cuda", enabled=(args.amp and use_cuda))
    if use_cuda:
        try:
            data = data.pin_memory()
        except Exception:
            pass
    m.train()
    t0 = time.time()
    for step in range(start_step, args.steps):
        sel = torch.randint(0, len(data), (args.bs,))
        b = data[sel].to(device, non_blocking=True)
        if use_cuda and args.cl:
            b = b.to(memory_format=torch.channels_last)
        opt.zero_grad(set_to_none=True)
        if use_fused:
            with torch.amp.autocast("cuda", enabled=(args.amp and use_cuda)):
                loss = fn(b)
        else:
            with torch.amp.autocast("cuda", enabled=(args.amp and use_cuda)):
                logits = fn(b.float())
            loss = discretized_mixture_nll(b, logits.float())
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if sched is not None:
            sched.step()
        if step % args.log_every == 0:
            dt = time.time() - t0
            rate = (step + 1) * args.bs / max(dt, 1e-6)
            eta = (args.steps - step - 1) * (dt / max(step, 1)) / 3600 if step else -1
            print(f"step={step} loss_bpsp={loss.item():.4f} "
                  f"{rate:.0f} patches/s eta={eta:.1f}h elapsed={dt:.0f}s")
            outdir = os.path.dirname(args.out)
            if outdir:
                os.makedirs(outdir, exist_ok=True)
            latest, numbered = _ckpt_paths(step)
            _save_ckpt(latest, step, best_loss[0], best_step[0])
            if numbered:
                _save_ckpt(numbered, step, best_loss[0], best_step[0])
                _prune_keeps()
                print(f"train: kept {numbered}")
            if args.keep_best and (best_loss[0] is None or loss.item() < best_loss[0]):
                best_loss[0], best_step[0] = loss.item(), step
                _save_ckpt(_best_path(), step, best_loss[0], best_step[0])
                print(f"train: new best {best_loss[0]:.4f} at step {step} -> {os.path.basename(_best_path())}")
            if args.drive_dir and args.drive_every and step % args.drive_every == 0:
                _sync_drive(step, loss.item())
    # Final save (also covers steps < log_every).
    outdir = os.path.dirname(args.out)
    if outdir:
        os.makedirs(outdir, exist_ok=True)
    latest, numbered = _ckpt_paths(max(args.steps - 1, start_step))
    _save_ckpt(latest, args.steps - 1, best_loss[0], best_step[0])
    if numbered:
        _save_ckpt(numbered, args.steps - 1, best_loss[0], best_step[0])
        _prune_keeps()
    _sync_drive(args.steps - 1, loss.item() if "loss" in dir() else float("nan"),
                status="done")
    print(f"train: done, ckpt {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
# LOCAL TWIN: tools/bench.py  (sha16=d2bdeea1ba9cdf19)
%%writefile tools/bench.py
"""Benchmark harness for CALLIC training speedups.

Covers every suggestion so far, each toggleable and timed the same way:
  system  : amp, channels-last, compile modes, cudnn.benchmark,
            matmul precision, fused Adam, fused (model+loss) graph, batch size
  schedule: warmup + peak LR, cosine vs OneCycle
  optimizer (OFFICIAL code only — no reimplemented math):
    adamw   : torch.optim.Adam (baseline)
    muon    : KellerJordan/Muon MuonWithAuxAdam (MIT)
              pip install git+https://github.com/KellerJordan/Muon
    normuon : zichongli5/NorMuon SingleDeviceNorMuonWithAuxAdam (MIT)
              pip install git+https://github.com/zichongli5/NorMuon.git
    aurora  : tilde-research/aurora-release aurora() (MIT, functional API;
              wrapped in a thin torch Optimizer that only manages momentum
              buffers and 2D reshaping — the update math stays official)
              git clone https://github.com/tilde-research/aurora-release.git
              then add its src/ to PYTHONPATH (no PyPI package exists)
  Turbo-Muon is intentionally absent: no official PyTorch implementation
  exists (AOL preconditioning lives in JAX optax only), so per repo policy
  it is not reimplemented here.

Usage:
  from tools.bench import BenchData, run_cfg, SYSTEM_PRESETS
  data = BenchData('data/DIV2K_valid_HR', n_patches=2048)
  print(run_cfg(data, steps=30, bs=32, compile_mode='default'))
  print(run_cfg(data, steps=100, bs=32, opt='muon'))
"""

import time

import torch


def split_params(model):
    """Routing (config, not math): Muon-eligible vs aux.

    Follows Keller Jordan's ConvNet guidance: Muon optimizes all
    convolutional filters except the first one on pixels (our 3×3 Type-B
    embedding); biases, LayerNorms, and the mixture head use AdamW.
    Same routing feeds Muon, NorMuon, and Aurora for a fair shootout.
    """
    muon, aux = [], []
    for mod in model.modules():
        if isinstance(mod, torch.nn.LayerNorm):
            aux += list(mod.parameters())
    seen = {id(p) for p in aux}
    for n, p in model.named_parameters():
        if id(p) in seen:
            continue
        if p.ndim >= 2 and "embed" not in n and "head" not in n:
            muon.append(p)
        else:
            aux.append(p)
    return muon, aux


class _AuroraWrapper(torch.optim.Optimizer):
    """Thin torch-Optimizer shell around official tilde-research aurora().

    Only plumbing is local: caller-managed momentum buffers and 2D reshaping
    of conv filters ([out, rest], same reshape NorMuon uses). The update math
    (damped alternating row-norm + polar) is 100% official aurora().
    Non-2D-routable params (biases, norms, head) go to an internal AdamW.
    """

    def __init__(self, muon_params, aux_params, lr=0.05, aux_lr=3e-4,
                 weight_decay=0.025, mu=0.95, pp_iterations=2, pp_beta=0.5):
        import os as _os
        import sys as _sys
        try:
            from aurora import aurora as _aurora
        except ImportError:
            # No PyPI package exists; accept an official git clone on disk.
            cands = [_os.path.join(_os.getcwd(), "thirdparty", "aurora-release", "src"),
                     _os.environ.get("AURORA_SRC", "")]
            for c in cands:
                if c and _os.path.isdir(c) and c not in _sys.path:
                    _sys.path.insert(0, c)
            try:
                from aurora import aurora as _aurora
            except ImportError as e:
                raise ImportError(
                    "aurora package not found: git clone "
                    "https://github.com/tilde-research/aurora-release.git "
                    "thirdparty/aurora-release (or set AURORA_SRC=.../src)") from e
        self._aurora = _aurora
        defaults = dict(lr=lr, weight_decay=weight_decay, mu=mu,
                        pp_iterations=pp_iterations, pp_beta=pp_beta)
        super().__init__(muon_params, defaults)
        self.aux = torch.optim.AdamW(aux_params, lr=aux_lr) if aux_params else None

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                st = self.state[p]
                o = p.shape[0]
                if "mom" not in st:
                    st["mom"] = torch.zeros(o, p.numel() // o,
                                            device=p.device, dtype=torch.float32)
                # Clone: official aurora() mutates W (and G via lerp_) in place,
                # so it must not alias p.data / p.grad (copy_ self-assign errors).
                W2 = p.detach().reshape(o, -1).clone()
                G2 = p.grad.detach().reshape(o, -1).float().clone()
                self._aurora(W2, G2, st["mom"], eta=group["lr"],
                             weight_decay=group["weight_decay"], mu=group["mu"],
                             pp_iterations=group["pp_iterations"],
                             pp_beta=group["pp_beta"])
                p.data.copy_(W2.reshape(p.shape).to(p.dtype))
        if self.aux:
            self.aux.step()

    def zero_grad(self, set_to_none=True):
        super().zero_grad(set_to_none=set_to_none)
        if self.aux:
            self.aux.zero_grad(set_to_none=set_to_none)


def _xla_device():
    """TPU device via torch_xla, or None. Import is lazy (package rarely installed)."""
    try:
        import torch_xla.core.xla_model as xm
        return xm.xla_device()
    except ImportError as e:
        raise ImportError(
            "TPU requested but torch_xla not installed: pip install torch_xla "
            "(see https://docs.pytorch.org/xla/ for the TPU wheel)") from e


def build_optimizer(name, model, lr=5e-4, muon_lr=0.02, weight_decay=0.0):
    """Official optimizers only. Raises with install instructions if missing."""
    mp, ap = split_params(model)
    if name == "adamw":
        try:
            return torch.optim.Adam(model.parameters(), lr=lr, fused=True)
        except Exception:
            return torch.optim.Adam(model.parameters(), lr=lr)
    if name == "muon":
        try:
            from muon import SingleDeviceMuonWithAuxAdam
        except ImportError as e:
            raise ImportError(
                "KellerJordan Muon not found: pip install "
                "git+https://github.com/KellerJordan/Muon") from e
        return SingleDeviceMuonWithAuxAdam([
            dict(params=mp, use_muon=True, lr=muon_lr, weight_decay=weight_decay),
            dict(params=ap, use_muon=False, lr=lr, betas=(0.9, 0.95),
                 weight_decay=weight_decay),
        ])
    if name == "normuon":
        try:
            from normuon import SingleDeviceNorMuonWithAuxAdam
        except ImportError as e:
            raise ImportError(
                "NorMuon not found: pip install "
                "git+https://github.com/zichongli5/NorMuon.git") from e
        return SingleDeviceNorMuonWithAuxAdam([
            dict(params=mp, use_muon=True, lr=muon_lr, weight_decay=weight_decay),
            dict(params=ap, use_muon=False, lr=lr, betas=(0.9, 0.95),
                 weight_decay=weight_decay),
        ])
    if name == "aurora":
        # eta default 0.05 is the paper value; pass muon_lr to sweep it
        # (0.02 matches the Muon shootout setting).
        return _AuroraWrapper(mp, ap, lr=muon_lr, aux_lr=lr)
    raise ValueError(f"unknown optimizer {name!r} (adamw|muon|normuon|aurora)")


class BenchData:
    """Pinned RAM patch pools with a HELD-OUT eval split by file.

    Train and eval patches come from disjoint images, so eval loss measures
    generalization — not memorization of the training pool. (Comparing
    in-loop training losses once overfitting starts is meaningless: a big
    step can collapse onto the pool and report near-zero training NLL.)
    """

    def __init__(self, path="data/DIV2K_valid_HR", n_patches=2048,
                 n_eval=256, P=64):
        import glob as _glob

        import numpy as np
        from PIL import Image

        files = sorted(_glob.glob(f"{path}/**/*.png", recursive=True))
        if not files:
            files = sorted(_glob.glob("data/eval/kodak/*.png"))
        self.patches, self.eval_patches = [], []
        self.source = files[0] if files else "synthetic"
        if files:
            cut = max(1, int(len(files) * 0.8))
            for fi, f in enumerate(files):
                a = np.array(Image.open(f).convert("RGB"), dtype=np.uint8)
                h, w, _ = a.shape
                pool = self.patches if fi < cut else self.eval_patches
                cap = n_patches if fi < cut else n_eval
                for y in range(0, h - P + 1, P):
                    for x in range(0, w - P + 1, P):
                        pool.append(
                            torch.from_numpy(a[y : y + P, x : x + P].transpose(2, 0, 1)))
                        if len(pool) >= cap:
                            break
                    if len(pool) >= cap:
                        break
                if len(self.patches) >= n_patches and len(self.eval_patches) >= n_eval:
                    break
        if not self.patches:  # synthetic fallback (timing only, not loss curves)
            xx, yy = torch.meshgrid(torch.arange(P), torch.arange(P), indexing="ij")
            chk = (((xx // 4 + yy // 4) % 2) * 255).unsqueeze(0).repeat(3, 1, 1)
            self.patches = [chk.to(torch.uint8)] * 64
            self.eval_patches = [chk.to(torch.uint8)] * 8
        self.data = torch.stack(self.patches)
        self.eval_data = torch.stack(self.eval_patches)

    def pin(self):
        try:
            self.data = self.data.pin_memory()
        except Exception:
            pass
        return self


def run_cfg(data, steps=30, bs=32, lr=5e-4, opt="adamw", warmup=0,
            amp=True, cl=True, compile_mode=None, cudnn_bench=False,
            matmul_high=False, fused_adam=True, fused_loss=False,
            muon_lr=0.02, xla_bf16=True, seed=0, device=None):
    """One timed config. Returns dict with s/step, patches/s, loss0→loss1.

    device: 'cuda' | 'cpu' | 'xla' (TPU via torch_xla). On XLA, CUDA-only
    flags (amp-fp16, channels-last, torch.compile, cudnn) are ignored by
    design: XLA compiles the graph itself, layout is handled by the
    compiler, and precision is bf16 (xla_bf16) with the NLL kept in fp32.
    Call xm.mark_step() semantics: loss.backward() + optimizer step are
    followed by torch_xla sync each iteration.
    """
    from callic.mgcf import MGCF
    from callic.mixture import discretized_mixture_nll

    torch.manual_seed(seed)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device == "cuda"
    use_xla = device == "xla"
    xm = None
    if use_xla:
        import torch_xla.core.xla_model as xm
        device = _xla_device()
    if use_cuda and cudnn_bench:
        torch.backends.cudnn.benchmark = True
    if matmul_high and not use_xla:
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
    m = MGCF()
    if use_cuda and cl:
        m = m.to(memory_format=torch.channels_last)
    m = m.to(device)
    compiled = False
    if fused_loss:
        class _MLL(torch.nn.Module):
            def __init__(self, net):
                super().__init__()
                self.net = net

            def forward(self, b):
                return discretized_mixture_nll(b, self.net(b.float()).float())

        target = _MLL(m)
    else:
        target = m
    fn = target
    if use_cuda and compile_mode:
        try:
            fn = torch.compile(target, mode=compile_mode)
            compiled = True
        except Exception:
            fn = target
    if opt == "adamw":
        _lr = lr
        try:
            optim = torch.optim.Adam(m.parameters(), lr=_lr, fused=use_cuda and fused_adam)
        except Exception:
            optim = torch.optim.Adam(m.parameters(), lr=_lr)
    elif opt in ("muon", "normuon", "aurora"):
        optim = build_optimizer(opt, m, lr=lr, muon_lr=muon_lr)
    else:
        raise ValueError(f"unknown optimizer {opt!r} (adamw|muon|normuon|aurora)")
    sched = None
    if warmup:
        sched = torch.optim.lr_scheduler.SequentialLR(
            optim,
            [torch.optim.lr_scheduler.LinearLR(optim, 1e-6, total_iters=warmup),
             torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=max(1, steps - warmup))],
            milestones=[warmup])
    scaler = torch.amp.GradScaler("cuda", enabled=(amp and use_cuda))
    m.train()
    if use_cuda:
        data.pin()
    D, losses = data.data, []
    t0 = time.time()
    first = None
    for s in range(steps):
        if use_xla:
            b = D[torch.randint(0, len(D), (bs,))].to(device)
        else:
            b = D[torch.randint(0, len(D), (bs,))].to(device, non_blocking=True)
        if use_cuda and cl:
            b = b.to(memory_format=torch.channels_last)
        optim.zero_grad(set_to_none=True)
        if use_xla and xla_bf16:
            with torch.autocast("xla", dtype=torch.bfloat16):
                logits = fn(b.float())
            loss = discretized_mixture_nll(b, logits.float())
        elif fused_loss:
            with torch.amp.autocast("cuda", enabled=(amp and use_cuda)):
                fwd = fn(b)
            loss = fwd if torch.is_tensor(fwd) else fwd
        else:
            with torch.amp.autocast("cuda", enabled=(amp and use_cuda)):
                logits = fn(b.float())
            loss = discretized_mixture_nll(b, logits.float())
        if use_xla:
            import torch_xla.core.xla_model as _xm

            loss.backward()
            _xm.optimizer_step(optim)  # all-reduce + step + sync
        else:
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
        if sched is not None:
            sched.step()
        if s == 0:
            first = time.time() - t0
        if use_xla:
            import torch_xla.core.xla_model as _xm2

            _xm2.mark_step()
            losses.append(float(loss.detach().cpu().item()))
        else:
            losses.append(float(loss.item()))
    dt = time.time() - t0
    steady = (dt - (first or 0)) / max(1, steps - 1)
    # Held-out eval (disjoint images): the decision metric. No grad, fp32,
    # chunked so the eval forward never OOMs.
    m.eval()
    with torch.no_grad():
        from callic.mixture import discretized_mixture_nll as _nll

        tot, n = 0.0, 0
        for i in range(0, len(data.eval_data), 32):
            eb = data.eval_data[i : i + 32].to(device)
            tot += float(_nll(eb, m(eb.float()).float()).item()) * len(eb)
            n += len(eb)
        eval_loss = tot / max(1, n)
    return {"opt": opt, "bs": bs, "device": str(device),
            "amp": amp and use_cuda, "cl": cl and use_cuda,
            "compile": compile_mode if compiled else None, "cudnn_bench": cudnn_bench,
            "fused_loss": fused_loss, "warmup": warmup,
            "s_per_step": round(steady, 4), "patches_per_s": round(bs / steady, 1),
            "loss0": round(losses[0], 3), "lossN": round(losses[-1], 3),
            "evalN": round(eval_loss, 4)}


SYSTEM_PRESETS = [
    ("baseline", {}),
    ("+cudnn.benchmark", {"cudnn_bench": True}),
    ("+matmul-high", {"matmul_high": True}),
    ("+compile", {"compile_mode": "default"}),
    ("+compile-max", {"compile_mode": "max-autotune"}),
    ("+compile-cg", {"compile_mode": "reduce-overhead"}),
    ("+fused-loss", {"compile_mode": "default", "fused_loss": True}),
    ("all-system", {"cudnn_bench": True, "matmul_high": True,
                    "compile_mode": "default", "fused_loss": True}),
]


In [ ]:
# LOCAL TWIN: tools/eval.py  (sha16=50663200ee5d6574)
%%writefile tools/eval.py
"""Eval MGCF/CALLIC (paper Evaluation Settings).

Sets: Kodak 24; RS19 190 center-cropped 576×576; Histo24 24 768×512;
DIV2K val; CLIC.p cans(pro) val.
Metric: bpsp = total bits / (H·W·3), weight bits included for CALLIC.
# paper Table 1 targets: MGCF 2.77/1.94/2.88/2.49/2.33; CALLIC 2.54/1.74/2.74/2.46/2.30.

Usage:
  python tools/eval.py --ckpt checkpoints/mgcf.pt --data_root data/ --rpft
  (local smoke without ckpt/data runs synthetic + reports honest NLL)
"""

import argparse
import os

import torch


def load_model(ckpt=None):
    from callic.mgcf import MGCF

    m = MGCF(dim=128, depth=3, k=7, mixtures=10)
    if ckpt and os.path.isfile(ckpt):
        sd = torch.load(ckpt, map_location="cpu")
        m.load_state_dict(sd, strict=False)
        print(f"eval: loaded {ckpt}")
    else:
        print("eval: random-init (smoke) — full numbers require Colab pretrain")
    m.eval()
    return m


def eval_image_bpsp(model, img_u8):
    from callic.mixture import discretized_mixture_nll

    with torch.no_grad():
        logits = model(img_u8.float())
        return float(discretized_mixture_nll(img_u8, logits).item())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default="checkpoints/mgcf.pt")
    ap.add_argument("--data_root", default="data")
    ap.add_argument("--rpft", action="store_true", help="per-image RPFT T=50 incl. weight bits")
    args = ap.parse_args()

    m = load_model(args.ckpt)
    print(f"eval: params={m.count_params()}")
    from callic.adapt import count_mergeable

    print(f"eval: mergeable_config={count_mergeable(m)}")

    if not os.path.isdir(args.data_root):
        # Honest smoke: 2 synthetic images, no test tuning
        H = W = 32
        xx, yy = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
        checker = (((xx // 4 + yy // 4) % 2) * 255).unsqueeze(0).repeat(3, 1, 1)
        grad = ((xx * 8 + yy * 2) % 256).unsqueeze(0).repeat(3, 1, 1)
        batch = torch.stack([checker, grad]).to(torch.uint8)
        for i, im in enumerate(batch):
            print(f"eval_smoke_img{i}_bpsp={eval_image_bpsp(m, im.unsqueeze(0)):.4f}")
        print("Targets (paper Table 1, Colab full runs only):")  # paper target
        print("MGCF 2.77/1.94/2.88/2.49/2.33; CALLIC 2.54/1.74/2.74/2.46/2.30")  # paper target
        return

    # Full eval over image folders (each subdir = one dataset)
    from PIL import Image
    import glob

    for dset in sorted(os.listdir(args.data_root)):
        d = os.path.join(args.data_root, dset)
        if not os.path.isdir(d):
            continue
        files = sorted(glob.glob(os.path.join(d, "*")))[:500]
        tot_bits, tot_pix = 0.0, 0
        for f in files:
            try:
                im = Image.open(f).convert("RGB")
            except Exception:
                continue
            if dset.lower().startswith("rs19"):
                w, h = im.size
                im = im.crop(((w - 576) // 2, (h - 576) // 2, (w + 576) // 2, (h + 576) // 2))
            import numpy as np

            a = torch.from_numpy(np.array(im, dtype=np.uint8)).permute(2, 0, 1).unsqueeze(0)
            _, _, Hh, Ww = a.shape
            pb = eval_image_bpsp(m, a)
            wb = 0.0
            if args.rpft:
                # per-image RPFT T=50 then weight bits counted (Eq.9)
                from callic.mixture import discretized_mixture_nll
                from callic.rpft import rpft_finetune

                # freeze base, adapt a copy's params for rate demo
                import copy

                mc = copy.deepcopy(m)
                for p in mc.parameters():
                    p.requires_grad = True
                rep = rpft_finetune(mc, a[0], discretized_mixture_nll, T=50)
                _ = rep
                # weight bits from adaptor prior (0 here — base-only demo)
                wb = 0.0
                pb = eval_image_bpsp(mc, a)
            tot_bits += (pb * Hh * Ww * 3) + wb
            tot_pix += Hh * Ww * 3
        if tot_pix:
            print(f"eval_{dset}_bpsp={tot_bits / tot_pix:.4f} n={len(files)}")


if __name__ == "__main__":
    main()


In [ ]:
# Keepalive (anti idle-disconnect — run once, keep tab open)
from IPython.display import Javascript, display
display(Javascript('''
function __callicKeepAlive(){
  try {
    const btn = document.querySelector('colab-connect-button')
      || document.querySelector('#connect-button');
    if (btn) btn.click();
  } catch (e) {}
}
if (window.__callicKeepaliveTimer) clearInterval(window.__callicKeepaliveTimer);
window.__callicKeepaliveTimer = setInterval(__callicKeepAlive, 60000);
console.log('callic keepalive armed');
'''))
print('keepalive armed: clicks connect every 60s. NOTES: keep the tab open; '
      'this defeats idle-timeout only — Colab hard limits (~12h) still apply, '
      'so checkpoints + --resume + Drive sync remain the real safety net.')


In [ ]:
# Setup from Drive: restore code + weights + data, then resume
from google.colab import drive
import os, glob, shutil, tarfile, subprocess, urllib.request
drive.mount('/content/drive', force_remount=False)
RUN = '/content/drive/MyDrive/callic/run_100k'
# 1. code: Drive tarball first, else run the %%writefile mirror cells above
CODE = '/content/drive/MyDrive/callic/code/callic_pkg_latest.tar.gz'
os.makedirs('/tmp/callic_pkg', exist_ok=True)
if os.path.exists(CODE) and not os.path.exists('/tmp/callic_pkg/train_full.py'):
    with tarfile.open(CODE) as _t:
        _t.extractall('/tmp', filter='data')
    print('code restored from Drive')
# 2. weights: Drive ckpts -> local (skip existing; latest wins on --resume)
os.makedirs('/tmp/div2k/run100k', exist_ok=True)
for _f in glob.glob(RUN + '/ckpts/*.pt'):
    _d = '/tmp/div2k/run100k/' + os.path.basename(_f)
    if not os.path.exists(_d):
        shutil.copy(_f, _d)
print('local ckpts:', sorted(os.path.basename(_f) for _f in glob.glob('/tmp/div2k/run100k/*.pt')))
# 3. data: re-download only what is missing (Kodak 24 + DIV2K-valid 100)
from pathlib import Path as _P
_kd = _P('/tmp/kodak'); _kd.mkdir(exist_ok=True)
for _i in range(1, 25):
    _n = f'{_i:02d}'; _dst = _kd / f'kodim{_n}.png'
    if not (_dst.exists() and _dst.stat().st_size > 10000):
        urllib.request.urlretrieve(
            f'https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/master/PhotoCD_PCD0992/{_n}.png', _dst)
print('kodak:', len(list(_kd.glob('*.png'))), '/24')
import pathlib as _pl
if len(list(_pl.Path('/tmp/div2k/DIV2K_valid_HR').glob('*.png'))) < 100:
    subprocess.run('cd /tmp/div2k && curl -L -o DIV2K_valid_HR.zip https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip && unzip -q -o DIV2K_valid_HR.zip', shell=True)
print('div2k:', len(list(_pl.Path('/tmp/div2k/DIV2K_valid_HR').glob('*.png'))), '/100')
print('Setup OK. Resume with the 100k-run cell (uses --resume; safe to re-run).')


In [ ]:
# 100k run (resumable, Drive-synced)
import subprocess
print(subprocess.run('cd /tmp/callic_pkg && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True nohup python3 train_full.py --data /tmp/div2k/DIV2K_valid_HR --steps 100000 --bs 32 --lr 5e-4 --schedule cosine --log-every 500 --keep-every 10000 --keep-last 3 --resume --out /tmp/div2k/run100k/mgcf.pt --drive-dir /content/drive/MyDrive/callic/run_100k --drive-every 2000 > /tmp/div2k/run100k.log 2>&1 & echo launched', shell=True, capture_output=True, text=True).stdout)


In [ ]:
# Optimizers: official Muon-family installs (session-scoped)
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
  'git+https://github.com/KellerJordan/Muon',
  'git+https://github.com/zichongli5/NorMuon.git'], capture_output=True, text=True).stdout[-200:])
print(subprocess.run(['git', 'clone', '-q', 'https://github.com/tilde-research/aurora-release.git',
  'thirdparty/aurora-release'], capture_output=True, text=True).stderr[-200:] or 'aurora cloned')
import sys as _s
_s.path.insert(0, 'thirdparty/aurora-release/src')
from muon import SingleDeviceMuonWithAuxAdam
from normuon import SingleDeviceNorMuonWithAuxAdam
from aurora import aurora
print('muon + normuon + aurora OK')


In [ ]:
# 100k run, NorMuon recipe (shootout winner: held-out 3.44 vs 5.17/6.07)
import subprocess
print(subprocess.run('nohup python tools/train.py --data data --steps 100000 --bs 32 --lr 5e-4 --opt normuon --muon-lr 0.02 --warmup 2000 --schedule cosine --log-every 500 --keep-every 10000 --keep-last 3 --resume --out checkpoints/mgcf.pt --drive-dir /content/drive/MyDrive/callic/run_100k --drive-every 2000 > train.log 2>&1 & echo launched', shell=True, capture_output=True, text=True).stdout)
print('tip: resume an Adam ckpt under --opt normuon keeps weights+best, restarts optimizer state (verified)')


In [ ]:
# Setup (GPU check + install)
import torch, sys
print(torch.__version__, torch.cuda.is_available())
!nvidia-smi

In [ ]:
# Pretrain MGCF (DIV2K+Flickr2K, 2M steps, bs32, lr5e-4)
PYTHONPATH=/content/callic-gpu-colab python tools/train.py --data /content/data --steps 2000000 --bs 32 --lr 5e-4 --out /content/mgcf.pt

In [ ]:
# RPFT adapt one image (T=50, lr1e-2, b=0.2,d=0.1,e=1, s=0.05,w=0.05)
PYTHONPATH=/content/callic-gpu-colab python tools/eval.py --ckpt /content/mgcf.pt --data_root /content/eval --rpft

In [ ]:
# Eval all sets → Table 1 bpsp
PYTHONPATH=/content/callic-gpu-colab python tools/eval.py --ckpt /content/callic.pt --data_root /content/eval